<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment40_Digital_Evidence_Hash_Integrity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# EXPERIMENT 2
# DIGITAL EVIDENCE HASH INTEGRITY
# Cryptographic Hashing, Tamper Detection and Hash-Set Matching
# ============================================================

import hashlib
import os
import tempfile

# Read files in 1 MB blocks
CHUNK = 1024 * 1024


# ============================================================
# 1. HASH A FILE
# ============================================================

def hash_file(path, algos=("md5", "sha1", "sha256")):

    hs = {
        algo: hashlib.new(algo)
        for algo in algos
    }

    with open(path, "rb") as f:

        while True:

            block = f.read(CHUNK)

            if not block:
                break

            for h in hs.values():
                h.update(block)

    return {
        algo: h.hexdigest()
        for algo, h in hs.items()
    }


# ============================================================
# 2. HASH BYTE DATA
# ============================================================

def hash_bytes(data, algo="sha256"):

    return hashlib.new(
        algo,
        data
    ).hexdigest()


# ============================================================
# 3. VERIFY FILE INTEGRITY
# ============================================================

def verify(path, expected, algo="sha256"):

    actual = hash_file(
        path,
        (algo,)
    )[algo]

    return (
        actual.lower() == expected.lower(),
        actual
    )


# ============================================================
# 4. TAMPER WITH ONE BYTE
# ============================================================

def tamper(path, offset=0, replacement=b"X"):

    with open(path, "r+b") as f:

        f.seek(offset)

        f.write(replacement)


# ============================================================
# 5. HASH-SET LOOKUP
# ============================================================

def match_hashset(digest, known_bad):

    known_bad_lower = {
        d.lower()
        for d in known_bad
    }

    return digest.lower() in known_bad_lower


# ============================================================
# 6. TEST CASES
# ============================================================

def run_tests():

    tmp = tempfile.mkdtemp()

    # --------------------------------------------------------
    # Create evidence file
    # --------------------------------------------------------

    evidence_file = os.path.join(
        tmp,
        "evidence.txt"
    )

    with open(
        evidence_file,
        "w"
    ) as f:

        f.write(
            "Seized from workstation WS-0142 "
            "on 2026-08-20.\n"
            "Invoice payment details.\n"
        )


    # Calculate original hashes

    original_hashes = hash_file(
        evidence_file
    )


    results = []


    # ========================================================
    # TC1 - HASH LENGTHS
    # ========================================================

    results.append(
        (
            "TC1 Digest lengths",
            len(original_hashes["md5"]) == 32
            and
            len(original_hashes["sha1"]) == 40
            and
            len(original_hashes["sha256"]) == 64
        )
    )


    # ========================================================
    # TC2 - DETERMINISM
    # ========================================================

    second_hash = hash_file(
        evidence_file
    )

    results.append(
        (
            "TC2 Determinism",
            second_hash["sha256"]
            ==
            original_hashes["sha256"]
        )
    )


    # ========================================================
    # TC3 - VERIFY ORIGINAL FILE
    # ========================================================

    verified, actual = verify(
        evidence_file,
        original_hashes["sha256"]
    )

    results.append(
        (
            "TC3 Verify unmodified file",
            verified
        )
    )


    # ========================================================
    # TC4 - TAMPER DETECTION
    # ========================================================

    tamper(
        evidence_file,
        offset=10,
        replacement=b"Z"
    )

    verified_after_tamper, new_hash = verify(
        evidence_file,
        original_hashes["sha256"]
    )

    results.append(
        (
            "TC4 Tamper detected",
            verified_after_tamper is False
            and
            new_hash != original_hashes["sha256"]
        )
    )


    # ========================================================
    # TC5 - SHA-256 EMPTY FILE TEST VECTOR
    # ========================================================

    empty_file = os.path.join(
        tmp,
        "empty.bin"
    )

    open(
        empty_file,
        "wb"
    ).close()

    empty_hash = hash_file(
        empty_file,
        ("sha256",)
    )["sha256"]

    expected_empty_hash = (
        "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
    )

    results.append(
        (
            "TC5 Empty-file test vector",
            empty_hash == expected_empty_hash
        )
    )


    # ========================================================
    # TC6 - SHA-256 'abc' TEST VECTOR
    # ========================================================

    abc_hash = hash_bytes(
        b"abc"
    )

    expected_abc_hash = (
        "ba7816bf8f01cfea414140de5dae2223"
        "b00361a396177a9cb410ff61f20015ad"
    )

    results.append(
        (
            "TC6 'abc' test vector",
            abc_hash == expected_abc_hash
        )
    )


    # ========================================================
    # TC7 - CHUNKED READING TEST
    # ========================================================

    big_file = os.path.join(
        tmp,
        "big.bin"
    )

    with open(
        big_file,
        "wb"
    ) as f:

        f.write(
            os.urandom(
                3 * 1024 * 1024
            )
        )

    with open(
        big_file,
        "rb"
    ) as f:

        whole_data = f.read()

    chunked_hash = hash_file(
        big_file,
        ("sha256",)
    )["sha256"]

    whole_file_hash = hash_bytes(
        whole_data
    )

    results.append(
        (
            "TC7 Chunked == whole-file",
            chunked_hash == whole_file_hash
        )
    )


    # ========================================================
    # TC8 - HASH-SET MATCHING
    # ========================================================

    bad_hash_1 = hash_bytes(
        b"malware-sample-A"
    )

    bad_hash_2 = hash_bytes(
        b"malware-sample-B"
    )

    known_bad = {
        bad_hash_1,
        bad_hash_2
    }

    match_found = match_hashset(
        hash_bytes(b"malware-sample-A"),
        known_bad
    )

    clean_match = match_hashset(
        hash_bytes(b"clean-file"),
        known_bad
    )

    results.append(
        (
            "TC8 Hash-set match",
            match_found is True
            and
            clean_match is False
        )
    )


    # ========================================================
    # DISPLAY RESULTS
    # ========================================================

    print("=" * 70)
    print("DIGITAL EVIDENCE HASHING - TEST RESULTS")
    print("=" * 70)

    for name, passed in results:

        print(
            f"{name:<35} -> "
            f"{'PASS' if passed else 'FAIL'}"
        )

    passed_count = sum(
        1
        for _, passed in results
        if passed
    )

    print("-" * 70)

    print(
        f"RESULT: {passed_count}/{len(results)} "
        "test cases passed"
    )

    print("=" * 70)

    return passed_count == len(results)


# ============================================================
# 7. RUN ALL TESTS
# ============================================================

run_tests()


# ============================================================
# 8. CREATE YOUR OWN EVIDENCE FILE
# ============================================================

print("\n")
print("=" * 70)
print("DIGITAL EVIDENCE HASH GENERATION")
print("=" * 70)

user_file = os.path.join(
    tempfile.gettempdir(),
    "colab_evidence.txt"
)

with open(
    user_file,
    "w"
) as f:

    f.write(
        "Digital evidence sample created for forensic examination.\n"
        "Evidence ID: EV-001\n"
        "Integrity verification experiment.\n"
    )


# Calculate MD5, SHA-1 and SHA-256

digests = hash_file(
    user_file
)


print("\nEvidence File:")
print(user_file)

print("\nMD5:")
print(digests["md5"])

print("\nSHA-1:")
print(digests["sha1"])

print("\nSHA-256:")
print(digests["sha256"])


# ============================================================
# 9. INTEGRITY VERIFICATION
# ============================================================

print("\n")
print("=" * 70)
print("INTEGRITY VERIFICATION")
print("=" * 70)

is_valid, calculated_hash = verify(
    user_file,
    digests["sha256"],
    "sha256"
)

print(
    "Original SHA-256 :",
    digests["sha256"]
)

print(
    "Calculated SHA-256:",
    calculated_hash
)

print(
    "Integrity Status  :",
    "VERIFIED - File is unchanged"
    if is_valid
    else
    "FAILED - File has changed"
)


# ============================================================
# 10. TAMPER DEMONSTRATION
# ============================================================

print("\n")
print("=" * 70)
print("SINGLE-BYTE TAMPER DEMONSTRATION")
print("=" * 70)

tamper(
    user_file,
    offset=10,
    replacement=b"X"
)

tampered_hash = hash_file(
    user_file,
    ("sha256",)
)["sha256"]

print(
    "Original SHA-256 :",
    digests["sha256"]
)

print(
    "Tampered SHA-256 :",
    tampered_hash
)

print(
    "Tamper Detection :",
    "DETECTED"
    if tampered_hash != digests["sha256"]
    else
    "NOT DETECTED"
)


# ============================================================
# 11. FINAL RESULT
# ============================================================

print("\n")
print("=" * 70)
print("EXPERIMENT 2 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "MD5, SHA-1 and SHA-256 hashing were performed."
)

print(
    "File integrity was verified successfully."
)

print(
    "Single-byte modification was detected."
)

print(
    "Hash-set matching was demonstrated."
)

print("=" * 70)

DIGITAL EVIDENCE HASHING - TEST RESULTS
TC1 Digest lengths                  -> PASS
TC2 Determinism                     -> PASS
TC3 Verify unmodified file          -> PASS
TC4 Tamper detected                 -> PASS
TC5 Empty-file test vector          -> PASS
TC6 'abc' test vector               -> PASS
TC7 Chunked == whole-file           -> PASS
TC8 Hash-set match                  -> PASS
----------------------------------------------------------------------
RESULT: 8/8 test cases passed


DIGITAL EVIDENCE HASH GENERATION

Evidence File:
/tmp/colab_evidence.txt

MD5:
c2342a09e68b9783986516ed3d7033bc

SHA-1:
8a2171c2f3f715596ce8036a4ca09e605db10dff

SHA-256:
dc6bec1260054d4d2246b7d6dde8882825a0366ee6ae6d8b6d47a8ef3cd5af6f


INTEGRITY VERIFICATION
Original SHA-256 : dc6bec1260054d4d2246b7d6dde8882825a0366ee6ae6d8b6d47a8ef3cd5af6f
Calculated SHA-256: dc6bec1260054d4d2246b7d6dde8882825a0366ee6ae6d8b6d47a8ef3cd5af6f
Integrity Status  : VERIFIED - File is unchanged


SINGLE-BYTE TAMPER DEMON